In [ ]:
pip install pandas pyarrow

In [ ]:
pip install endee

In [1]:
pip show endee

Name: endee
Version: 0.1.13
Summary: Endee is the Next-Generation Vector Database for Scalable, High-Performance AI
Home-page: https://endee.io
Author: Endee Labs
Author-email: dev@endee.io
License: 
Location: /home/debian/latest_VDB/VectorDBBench/venv2/lib/python3.11/site-packages
Requires: httpx, msgpack, numpy, orjson, pydantic, requests
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [2]:
base_path = "/home/debian/latest_VDB/VectorDBBench/vectordataset"
dataset_folder = "cohere/cohere_medium_1m"

In [3]:
#For checking parquet file contents
import pyarrow.parquet as pq
import os

file_name = "shuffle_train.parquet"  #input

# Build full path
file_path = os.path.join(base_path, dataset_folder, file_name)


parquet_file = pq.ParquetFile(file_path)

# Read only first batch of rows
first_batch = next(parquet_file.iter_batches(batch_size=5))
preview = first_batch.to_pandas()

for col in preview.columns:
    if preview[col].dtype == object and isinstance(preview[col].iloc[0], list):
        preview[col] = preview[col].apply(lambda x: x[:5] if x is not None else x)

print(preview)

       id                                                emb
0  322406  [0.19600096, -0.5270862, -0.29519123, 0.429556...
1  337610  [0.32829463, -0.055560444, -0.06252914, -0.101...
2  714728  [-0.054155815, 0.009554057, 0.24696879, -0.142...
3  354737  [0.2501778, 0.017853737, -0.43795395, 0.522256...
4  290979  [0.3444257, -0.62243223, -0.20940691, -0.08620...


In [4]:
from endee import Endee
client = Endee(token="localtest")
client.set_base_url("http://148.113.54.173:8080/api/v1")

In [5]:
#For checking indexes
for obj in client.list_indexes().get('indexes'):
    print(obj['name'])
    print(obj['total_elements'])
    print('\t')

test_1M_labelfilter_int16
1000000
	
test_1M_labelfilter_int16_latest_master
1000000
	
test_1M_labelfilter_latest_master_stability1
1000000
	
test_1M_labelfilter_latest_master_stability2
990000
	
test_1M_normal
1000000
	
test_1M_normal_latest_master
1000000
	
test_1M_numfilter_int16
1000000
	
test_1M_numfilter_int16_latest_master
1000000
	
test_1M_numfilter_latest_master_stability1
1000000
	
test_1M_numfilter_latest_master_stability2
1000000
	
test_1M_numfilter_latest_master_stability3
990000
	
test_1M_numfilter_latest_master_stability_bmw
1000000
	
test_1M_numfilter_latest_master_stability_vaib
1000000
	
test_1M_numfilter_latest_master_stability_vaib2
200000
	
test_1M_vaib_reupsertcheck
300000
	
test_1M_vaib_reupsertcheck_new1
262392
	


In [6]:
#give the index name
index_name = "test_1M_vaib_reupsertcheck_new1"
index = client.get_index(index_name)

In [8]:
index.describe()

{'name': 'test_1M_vaib_reupsertcheck_new1',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 262392,
 'precision': 'int16',
 'M': 16}

In [8]:
#For querying (int filter)- range filter
import pyarrow.parquet as pq
import os

#inputs
int_rate_query = "99p"   # change to 1p, 50p, 80p, 99p
top_k = 30

# Mode 1: Single query id
# Mode 2: Find all query ids with recall < 1
mode = 2
query_id = 457    # only used in mode 1

# Map int_rate_query to filter range
range_map = {
    "1p":  [10000, 1000000],
    "50p": [500000, 1000000],
    "80p": [800000, 1000000],
    "99p": [990000, 1000000],
}
filter_range = range_map[int_rate_query]

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
test_parquet_path = os.path.join(dataset_path, "test.parquet")
gt_parquet_path = os.path.join(dataset_path, f"neighbors_int_{int_rate_query}.parquet")

# Read parquet files
test_df = pq.ParquetFile(test_parquet_path).read().to_pandas()
gt_df = pq.ParquetFile(gt_parquet_path).read().to_pandas()

def run_query(query_id):
    query_row = test_df[test_df["id"] == query_id]
    if query_row.empty:
        print(f"Query ID {query_id} not found in test.parquet")
        return None

    query_vector = query_row["emb"].values[0]

    gt_row = gt_df[gt_df["id"] == query_id]
    if gt_row.empty:
        print(f"Query ID {query_id} not found in ground truth file")
        return None

    ground_truth = gt_row["neighbors_id"].values[0][:top_k]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        filter=[{"id": {"$range": filter_range}}],
        include_vectors=False,
    )
    returned_ids = [int(r["id"]) for r in results]

    intersection = len(set(returned_ids) & set(ground_truth))
    recall = intersection / len(ground_truth)

    return returned_ids, ground_truth, recall


if mode == 1:
    result = run_query(query_id)
    if result:
        returned_ids, ground_truth, recall = result
        print(f"Query ID: {query_id}")
        print(f"Returned IDs: {','.join(map(str, returned_ids))}")
        print(f"Ground Truth: {','.join(map(str, ground_truth))}")
        print(f"Recall: {recall:.4f}")

elif mode == 2:
    print(f"Finding all query IDs with recall < 1.0 ...\n")
    low_recall_ids = []
    all_recalls = []

    for qid in test_df["id"].values:
        result = run_query(qid)
        if result:
            returned_ids, ground_truth, recall = result
            all_recalls.append(recall)
            if recall < 1.0:
                low_recall_ids.append(qid)
                print(f"Query ID: {qid}")
                print(f"Returned IDs: {','.join(map(str, returned_ids))}")
                print(f"Ground Truth: {','.join(map(str, ground_truth))}")
                print(f"Recall: {recall:.4f}\n")

    print(f"Total query IDs with recall < 1.0: {len(low_recall_ids)}")
    print(f"IDs: {low_recall_ids}")
    print(f"Average Recall across all {len(all_recalls)} queries: {sum(all_recalls)/len(all_recalls):.4f}")

Finding all query IDs with recall < 1.0 ...

Query ID: 0
Returned IDs: 992169,992979,997002,993717,997637,997918,997822,993288,999453,992790,995401,994922,993289,994819,990399,993738,995994,998530,996438,991266,993032,999587,994334,991150,994964,994409,995701,995708,994804,999450
Ground Truth: 995888,992169,992979,997002,993717,991151,997637,997918,997822,993288,992933,999453,992790,995401,994922,993289,997852,994819,990399,993478,993738,995994,996777,998530,996438,991266,993412,990337,993032,999587
Recall: 0.7333

Query ID: 1
Returned IDs: 997420,995382,990103,993165,996814,996219,996878,992878,998104,994051,994199,990678,995438,992193,994119,997997,992780,993088,997331,992731,996811,998691,999191,991031,995783,999244,993324,996121,995562,993929
Ground Truth: 997420,995382,990103,993165,996814,996219,996878,992878,998104,994051,994199,990678,996062,995438,993727,992193,994119,997997,999441,996830,992780,993088,997331,990284,992731,997602,997098,996811,998691,999191
Recall: 0.7667

Que

In [6]:
#For querying (int filter)- gte filter
import pyarrow.parquet as pq
import os

#inputs
int_rate_query = "99p"   # change to 1p, 50p, 80p, 99p
top_k = 30

# Mode 1: Single query id
# Mode 2: Find all query ids with recall < 1
mode = 2
query_id = 457    # only used in mode 1

# Map int_rate_query to filter range
range_map = {
    "1p":  [10000, 1000000],
    "50p": [500000, 1000000],
    "80p": [800000, 1000000],
    "99p": [990000, 1000000],
}
filter_range = range_map[int_rate_query]

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
test_parquet_path = os.path.join(dataset_path, "test.parquet")
gt_parquet_path = os.path.join(dataset_path, f"neighbors_int_{int_rate_query}.parquet")

# Read parquet files
test_df = pq.ParquetFile(test_parquet_path).read().to_pandas()
gt_df = pq.ParquetFile(gt_parquet_path).read().to_pandas()

def run_query(query_id):
    query_row = test_df[test_df["id"] == query_id]
    if query_row.empty:
        print(f"Query ID {query_id} not found in test.parquet")
        return None

    query_vector = query_row["emb"].values[0]

    gt_row = gt_df[gt_df["id"] == query_id]
    if gt_row.empty:
        print(f"Query ID {query_id} not found in ground truth file")
        return None

    ground_truth = gt_row["neighbors_id"].values[0][:top_k]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        filter=[{"id": {"$gte": filter_range[0]}}],
        include_vectors=False,
    )
    returned_ids = [int(r["id"]) for r in results]

    intersection = len(set(returned_ids) & set(ground_truth))
    recall = intersection / len(ground_truth)

    return returned_ids, ground_truth, recall


if mode == 1:
    result = run_query(query_id)
    if result:
        returned_ids, ground_truth, recall = result
        print(f"Query ID: {query_id}")
        print(f"Returned IDs: {','.join(map(str, returned_ids))}")
        print(f"Ground Truth: {','.join(map(str, ground_truth))}")
        print(f"Recall: {recall:.4f}")

elif mode == 2:
    print(f"Finding all query IDs with recall < 1.0 ...\n")
    low_recall_ids = []
    all_recalls = []

    for qid in test_df["id"].values:
        result = run_query(qid)
        if result:
            returned_ids, ground_truth, recall = result
            all_recalls.append(recall)
            if recall < 1.0:
                low_recall_ids.append(qid)
                print(f"Query ID: {qid}")
                print(f"Returned IDs: {','.join(map(str, returned_ids))}")
                print(f"Ground Truth: {','.join(map(str, ground_truth))}")
                print(f"Recall: {recall:.4f}\n")

    print(f"Total query IDs with recall < 1.0: {len(low_recall_ids)}")
    print(f"IDs: {low_recall_ids}")
    print(f"Average Recall across all {len(all_recalls)} queries: {sum(all_recalls)/len(all_recalls):.4f}")

Finding all query IDs with recall < 1.0 ...

Query ID: 1
Returned IDs: 997420,995382,990103,993165,996814,996219,996878,992878,998104,994051,994199,990678,996062,995438,993727,992193,994119,997997,999441,992780,993088,997331,990284,992731,997602,997098,996811,998691,999191,991031
Ground Truth: 997420,995382,990103,993165,996814,996219,996878,992878,998104,994051,994199,990678,996062,995438,993727,992193,994119,997997,999441,996830,992780,993088,997331,990284,992731,997602,997098,996811,998691,999191
Recall: 0.9667

Query ID: 2
Returned IDs: 996312,991260,995701,992330,996592,993492,997449,997913,992546,997976,990231,990936,990949,995014,994100,996881,992692,992969,993853,993005,996267,994778,991045,994757,992931,990957,999647,992961,990109,993970
Ground Truth: 996312,991260,995701,992330,996592,993492,997449,997913,992546,997976,990231,990936,993517,990949,995014,994100,996881,992692,992969,993853,993005,996267,994778,991045,994757,992931,990957,999647,992961,990109
Recall: 0.9667

Que

In [10]:
index.describe()

{'name': 'test_1M_numfilter_latest_master_stability_vaib2',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 200000,
 'precision': 'int16',
 'M': 16}

In [ ]:
#For deleting (int filter) - range operator
int_rate_delete = "1p"  # change to 1p, 50p, 80p, 99p

# Map int_rate to range
range_map = {
    "1p":  [0, 9999],
    "50p": [0, 499999],
    "80p": [0, 799999],
    "99p": [0, 989999],
}
filter_range = range_map[int_rate_delete]

result = index.delete_with_filter([{"id": {"$range": filter_range}}])
print(f"Deleted vectors with id range: {filter_range}")
print(result)

In [9]:
#For deleting (int filter) - gte
int_rate_delete = "80p"  # change to 1p, 50p, 80p, 99p

# Map int_rate to lower bound (gte value)
range_map = {
    "1p":  10000,
    "50p": 500000,
    "60p": 600000,
    "70p": 700000,
    "80p": 800000,
    "99p": 990000,
}
gte_value = range_map[int_rate_delete]

result = index.delete_with_filter([{"id": {"$lt": gte_value}}])
print(f"Deleted vectors with id < {gte_value}")
print(result)

Deleted vectors with id < 800000
800000 vectors deleted


In [8]:
index.describe()

{'name': 'test_1M_vaib_reupsertcheck',
 'space_type': 'cosine',
 'dimension': 768,
 'sparse_dim': 0,
 'is_hybrid': False,
 'count': 300000,
 'precision': 'int16',
 'M': 16}

In [8]:
#For reinserting deleted vectors (int filter)
import pyarrow.parquet as pq
import os
import time

# User inputs
int_rate_insert = "80p"  # change to 1p, 50p, 80p, 99p
batch_size = 1000

# Map int_rate to range
range_map = {
    "1p":  [0, 9999],
    "50p": [0, 499999],
    "60p": [0, 599999],
    "70p": [0, 699999],
    "80p": [0, 799999],
    "99p": [0, 989999],
}
filter_range = range_map[int_rate_insert]
low, high = filter_range

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
emb_path = os.path.join(dataset_path, "shuffle_train.parquet")

# Process shuffle_train in batches to avoid RAM issues
emb_file = pq.ParquetFile(emb_path)
total_inserted = 0

start = time.perf_counter()
for batch in emb_file.iter_batches(batch_size=batch_size):
    batch_df = batch.to_pandas()

    # Filter by id range
    batch_df = batch_df[(batch_df["id"] >= low) & (batch_df["id"] <= high)]

    if batch_df.empty:
        continue

    batch_vectors = []
    for _, row in batch_df.iterrows():
        vector_data = {
            "id": str(row["id"]),
            "vector": row["emb"],
            "meta": {"id": row["id"]},
            "filter": {
                "id": row["id"],
            }
        }
        batch_vectors.append(vector_data)

    index.upsert(batch_vectors)
    total_inserted += len(batch_vectors)
    print(f"Upserted {len(batch_vectors)} vectors, total so far: {total_inserted}")
end = time.perf_counter()
print(f"Done! Total inserted: {total_inserted} vectors with id range {filter_range}")
print("Time taken:", end-start)

Upserted 798 vectors, total so far: 798
Upserted 812 vectors, total so far: 1610
Upserted 785 vectors, total so far: 2395
Upserted 813 vectors, total so far: 3208
Upserted 789 vectors, total so far: 3997
Upserted 827 vectors, total so far: 4824
Upserted 796 vectors, total so far: 5620
Upserted 804 vectors, total so far: 6424
Upserted 793 vectors, total so far: 7217
Upserted 787 vectors, total so far: 8004
Upserted 794 vectors, total so far: 8798
Upserted 797 vectors, total so far: 9595
Upserted 797 vectors, total so far: 10392
Upserted 817 vectors, total so far: 11209
Upserted 787 vectors, total so far: 11996
Upserted 802 vectors, total so far: 12798
Upserted 800 vectors, total so far: 13598
Upserted 786 vectors, total so far: 14384
Upserted 811 vectors, total so far: 15195
Upserted 772 vectors, total so far: 15967
Upserted 816 vectors, total so far: 16783
Upserted 796 vectors, total so far: 17579
Upserted 811 vectors, total so far: 18390
Upserted 767 vectors, total so far: 19157
Upser

ConnectionError: HTTPConnectionPool(host='148.113.54.173', port=8080): Max retries exceeded with url: /api/v1/index/test_1M_vaib_reupsertcheck_new1/vector/insert (Caused by NewConnectionError("HTTPConnection(host='148.113.54.173', port=8080): Failed to establish a new connection: [Errno 111] Connection refused"))

In [ ]:
index.describe()